[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/10_iterate_0p6b.ipynb)

# 10 · 一键实验：Qwen3-0.6B + Dolci（训练 → 评测 → 诊断）

**提交分支 `submission-yy726` 的迭代 notebook。** 一个 T4 runtime 里跑完一轮：
准备 Dolci 数据 → 去污染 → QLoRA SFT → 合并到本地 → 用锁定配置评测 GPQA → 输出诊断 JSON。

| ID | 配置 | 要回答的问题 |
|---|---|---|
| `E0_base` | Qwen3-0.6B-Base，不训练 | 起点分数；Base 会不会按格式作答 |
| `E1_think_sft` | Dolci-Think-SFT，math/science，5k 条，≤4k token | 长思维链 SFT 在 4096 token 预算下有没有用 |
| `E2_instruct_sft` | Dolci-Instruct-SFT，同样的过滤和数量 | 短回答风格 vs 长思维链 |
| `E3_think_strip` | 同 E1 的数据，但去掉 `<think>`（`strip_think=True`） | 同一批题目，只换答案风格 |

用法：在"实验配置"那一格把 `EXP_NAME` 改成上表里的一个 → **Runtime → Run all**。
跑完后把最后一格打印的 JSON **整段复制贴回给 Claude**（只有聚合数字，没有 GPQA 题目）。

迭代时 `SEEDS = "0"`（1 个 seed，噪声约 ±3 分）；确定最终方案后改成 `"0 1 2 3"` 再跑一次。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆比赛仓库（脚本都在里面） / clone the hackathon repo (branch: submission-yy726)
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q

In [ ]:
# 训练 + 评测装在同一个 runtime：照搬 Unsloth 官方 GRPO notebook 的 T4 安装方式（vllm==0.11.2），
# 再加上与之兼容的 lm-eval==0.4.11（和 main 分支锁定的评测版本一致）。约 5 分钟。
# 如果这里报依赖冲突：改用两个 runtime（02 训练并推到 Hub，01 评测）。
!pip install --upgrade -qqq uv
try:
    import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except Exception:
    _numpy, _pil = "numpy", "pillow"
!uv pip install -qqq --upgrade vllm==0.11.2 {_numpy} {_pil} torchvision bitsandbytes xformers unsloth triton
!uv pip install -qqq transformers==4.57.6 lm-eval==0.4.11
!uv pip install -qqq --no-deps trl==0.22.2
import importlib.metadata as md
VERSIONS = {}
for p in ["unsloth", "vllm", "lm-eval", "trl", "peft", "transformers", "torch"]:
    try:
        VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError:
        VERSIONS[p] = None
print(VERSIONS)

In [ ]:
# GPQA 是 gated 数据集：先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款，
# 再用有 read（提交模型需要 write）权限的 token 登录。
# 推荐：Colab 左侧 🔑 Secrets 里添加 HF_TOKEN，这里就会自动读取。
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# ==== 实验配置 / experiment config ====
EXP_NAME = "E0_base"            # E0_base | E1_think_sft | E2_instruct_sft | E3_think_strip
SEEDS    = "0"                  # 迭代用 "0"；最终提交用 "0 1 2 3"
SMOKE    = False                # True = 200 条数据 / 30 步训练 / 评测只跑 20 题（检查流程用，分数无效）

COMMON = dict(
    init_model="unsloth/Qwen3-0.6B-Base-unsloth-bnb-4bit", base_model="Qwen/Qwen3-0.6B-Base",
    include=["math", "science", "stem"], exclude=["safety", "wildguard"],
    n=5000, max_tokens=4096, strip_think=False,
    lora_r=32, lr=2e-4, epochs=1, batch=4, grad_accum=4,
)
PRESETS = {
    "E0_base":         dict(train=False),
    "E1_think_sft":    dict(train=True, dataset="allenai/Dolci-Think-SFT-7B"),
    "E2_instruct_sft": dict(train=True, dataset="allenai/Dolci-Instruct-SFT-7B"),
    "E3_think_strip":  dict(train=True, dataset="allenai/Dolci-Think-SFT-7B", strip_think=True),
}
CFG = {**COMMON, **PRESETS[EXP_NAME]}
if SMOKE:   # 快速验证：小数据、小 shuffle buffer（流式读取更快开始）
    CFG.update(n=200)
SHUF = 1000 if SMOKE else 10000
OUT = f"outputs/{EXP_NAME}"
# 给 `!` shell 命令用的简单变量
DATASET, INIT_MODEL, BASE_MODEL = CFG.get("dataset", ""), CFG["init_model"], CFG["base_model"]
N_ROWS, MAX_TOK = CFG["n"], CFG["max_tokens"]
print(CFG)

## 1. 数据（只在训练实验里执行）

第一次用某个数据集时，可以先运行下一格看看有哪些来源，再调整 `include` / `exclude`。

In [ ]:
if CFG["train"]:
    !python scripts/prepare_dolci_sft.py --dataset {DATASET} --out /tmp/x.jsonl --list-sources 20000

In [ ]:
import json, os
os.makedirs(OUT, exist_ok=True)
DATA_STATS = {}
if CFG["train"]:
    inc = " ".join(CFG["include"]); exc = " ".join(CFG["exclude"])
    strip = "--strip-think" if CFG["strip_think"] else ""
    !python scripts/prepare_dolci_sft.py --dataset {DATASET} --out {OUT}/sft.jsonl --n {N_ROWS} --tokenizer {INIT_MODEL} --max-tokens {MAX_TOK} --include {inc} --exclude {exc} {strip} --shuffle-buffer {SHUF}
    !python scripts/decontam.py --train-jsonl {OUT}/sft.jsonl --out {OUT}/decontam.json > /dev/null
    rep = json.load(open(f"{OUT}/decontam.json"))
    bad = set(rep["flagged_row_indices"])
    rows = [l for i, l in enumerate(open(f"{OUT}/sft.jsonl")) if i not in bad]
    open(f"{OUT}/sft_clean.jsonl", "w").writelines(rows)
    toks = [json.loads(l)["n_tokens"] for l in rows]
    DATA_STATS = dict(rows=len(rows), dropped_decontam=len(bad), overlap_rate=rep["overlap_rate"],
                      total_tokens=sum(toks), mean_tokens=round(sum(toks) / max(len(toks), 1)))
print(DATA_STATS)

## 2. QLoRA SFT

In [ ]:
import time, gc, torch
TRAIN_STATS = {}
EVAL_MODEL = CFG["base_model"]          # E0：直接评测原始 fp16 base

if CFG["train"]:
    from unsloth import FastLanguageModel
    from datasets import load_dataset
    from trl import SFTTrainer, SFTConfig

    model, tok = FastLanguageModel.from_pretrained(
        model_name=CFG["init_model"], max_seq_length=CFG["max_tokens"], load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=CFG["lora_r"], lora_alpha=2 * CFG["lora_r"], lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=0)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    ds = load_dataset("json", data_files=f"{OUT}/sft_clean.jsonl", split="train")
    ds = ds.remove_columns([c for c in ds.column_names if c not in ("prompt", "completion")])
    trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=ds, args=SFTConfig(
        output_dir=f"{OUT}/ckpt", per_device_train_batch_size=CFG["batch"],
        gradient_accumulation_steps=CFG["grad_accum"], learning_rate=CFG["lr"],
        num_train_epochs=CFG["epochs"], max_steps=30 if SMOKE else -1,
        warmup_ratio=0.03, lr_scheduler_type="cosine", max_length=CFG["max_tokens"],
        fp16=True, bf16=False, logging_steps=10, save_strategy="no", report_to="none", seed=0))
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    res = trainer.train()
    dt = time.time() - t0
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    TRAIN_STATS = dict(
        hours=round(dt / 3600, 3), peak_mem_gib=round(torch.cuda.max_memory_allocated() / 2**30, 2),
        tok_per_s=round(DATA_STATS.get("total_tokens", 0) * CFG["epochs"] / dt) if not SMOKE else None,
        steps=trainer.state.global_step, loss_first=losses[0] if losses else None,
        loss_last=losses[-1] if losses else None, loss_curve=losses[:: max(1, len(losses) // 20)])
    print(TRAIN_STATS)

    EVAL_MODEL = f"{OUT}/merged"
    model.save_pretrained_merged(EVAL_MODEL, tok, save_method="merged_16bit")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
print("eval model:", EVAL_MODEL)

## 3. 锁定配置评测 GPQA

In [ ]:
# 训练进程还占着一点显存，所以 vLLM 只申请 60%（0.6B 足够）。这个参数只影响显存，不影响分数。
limit = "LIMIT=20" if SMOKE else ""
t0 = time.time()
!GPU_MEM_UTIL=0.6 {limit} bash scripts/eval_gpqa.sh {EVAL_MODEL} results/{EXP_NAME} "{SEEDS}" vllm
EVAL_HOURS = round((time.time() - t0) / 3600, 3)

In [ ]:
!python scripts/aggregate_results.py results/{EXP_NAME} --model {EVAL_MODEL} --team yy726 --base-model {BASE_MODEL} --out {OUT}/scores.json > /dev/null
!python scripts/analyze_samples.py results/{EXP_NAME} --tokenizer {EVAL_MODEL} --out {OUT}/diag.json > /dev/null
scores = json.load(open(f"{OUT}/scores.json"))["scores"]
diag = json.load(open(f"{OUT}/diag.json"))
REPORT = dict(exp=EXP_NAME, smoke=SMOKE, seeds=SEEDS, config=CFG, data=DATA_STATS, train=TRAIN_STATS,
              eval_hours=EVAL_HOURS, scores=scores, diagnostics=diag, versions=VERSIONS)
json.dump(REPORT, open(f"{OUT}/report.json", "w"), indent=1)
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii=False))

## 4.（可选）满意之后：推到 Hub 并生成提交 JSON

确定最终方案后：先把 `SEEDS` 改成 `"0 1 2 3"` 重跑上面的评测，再运行下面这一格。

In [ ]:
PUSH = False
HUB_REPO = "YOUR_HF_USERNAME/qwen3-0.6b-dolci-" + EXP_NAME.lower()
if PUSH and CFG["train"]:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HUB_REPO, exist_ok=True, private=False)
    api.upload_folder(folder_path=EVAL_MODEL, repo_id=HUB_REPO)
    !python scripts/aggregate_results.py results/{EXP_NAME} --model {HUB_REPO} --team yy726 --base-model {BASE_MODEL} --out submissions/yy726.json
    print("填完 submissions/yy726.json 里的 TODO，然后提交到这个分支 / 开 PR。")